# GANForge: Custom DCGAN Fine-Tuning on Rangoli Art
### Google Colab Training Pipeline with Phase 5 FID Evaluation

This notebook provides a cloud training environment for the **Deep Convolutional Generative Adversarial Network (DCGAN)** using Google Colab.
When a GPU runtime is enabled, TensorFlow accelerates adversarial training compared to local CPU execution.

**Hardware Recommendation:**
1. Go to menu: **Runtime** -> **Change runtime type**
2. Hardware accelerator: Select **T4 GPU** (if available; otherwise standard CPU runtime will execute)
3. Click **Save**

In [ ]:
# Step 1: Verify Hardware Accelerator
!nvidia-smi

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n[+] TensorFlow successfully detected GPU: {gpus}")
else:
    print("\n[-] Warning: No GPU detected. Check Runtime -> Change runtime type -> T4 GPU.")

### Step 2: Environment Setup & Code Sync

In [ ]:
# Configure repository URL
import os
from pathlib import Path

REPO_URL = "https://github.com/samarth092005/GANForge.git"

if not Path("GANForge").exists() and not Path("data").exists():
    print(f"[*] Cloning repository from: {REPO_URL}")
    !git clone {REPO_URL}
    if Path("GANForge").exists():
        %cd GANForge
elif Path("GANForge").exists():
    %cd GANForge
    !git pull
else:
    print("[+] Already inside GANForge workspace directory.")

!ls -la

In [ ]:
# Install supplemental dependencies while preserving Colab's pre-configured TensorFlow GPU environment
# Note: Do NOT blindly install requirements.txt to avoid overwriting the GPU-tailored TensorFlow runtime
!pip install -q groq python-dotenv streamlit
print("[+] Supplemental dependencies installed (TensorFlow GPU environment preserved)!")

import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"Active GPU Accelerators: {gpus if gpus else 'None (Running on CPU)'}")

### Step 3: Dataset Ingestion & Preprocessing
Validates, normalizes to $[-1, 1]$, and splits the 300-image Rangoli dataset into 80% train (240 images) and 20% validation (60 images).

In [ ]:
!python -m data.prepare_rangoli_dataset --count 300

### Step 4: Run DCGAN Training Loop with Phase 5 FID Evaluation
Executes custom `tf.GradientTape` adversarial training loop with integrated per-epoch evaluation:
- **FID Reference**: Evaluated strictly against the 60 real validation images (`val_images.npy`).
- **Generator Evaluation**: Evaluates 60 synthetic samples generated from a fixed latent noise vector.
- **Feature Extraction**: Uses pre-trained InceptionV3 (2048-D pooled features) initialized once before training.
- **Visual Milestones**: Exports sample grids at epochs 1, 10, 25, 50.
- **Checkpointing**: Persists checkpoints and metrics (including FID) after every epoch.

> **Sample Size & Approximation Note**:
> Evaluating FID with 60 validation samples provides an approximate benchmark indicator for tracking relative training dynamics. Standard absolute FID publications traditionally use 10k-50k samples.

In [ ]:
!python -m training.train --epochs 50 --batch-size 32

### Step 5: Visualize Loss Curves & Convergence Dynamics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_df = pd.read_csv("outputs/metrics/training_history.csv")

plt.figure(figsize=(10, 5))
plt.plot(history_df['epoch'], history_df['generator_loss'], label='Generator Loss', color='royalblue', lw=2)
plt.plot(history_df['epoch'], history_df['discriminator_loss'], label='Discriminator Loss', color='crimson', lw=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (Binary Cross-Entropy)', fontsize=12)
plt.title('GANForge - DCGAN Adversarial Loss Dynamics (50 Epochs)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("outputs/metrics/loss_curve.png", dpi=150)
plt.show()

### Step 6: Display Fréchet Inception Distance (FID) Curve
Tracks distributional convergence across epochs. Lower FID scores indicate higher realism and closer alignment with real Rangoli validation features.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_df = pd.read_csv("outputs/metrics/training_history.csv")

if "fid_score" in history_df.columns and history_df["fid_score"].notna().any():
    plt.figure(figsize=(10, 5))
    plt.plot(
        history_df['epoch'],
        history_df['fid_score'],
        label='FID Score (Validation Split - 60 samples)',
        color='darkorange',
        lw=2.5,
        marker='o',
        markersize=4,
    )
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Fréchet Inception Distance (FID)', fontsize=12)
    plt.title('GANForge - FID Quality Progression (Lower is Better)', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig("outputs/metrics/fid_curve.png", dpi=150)
    plt.show()

    best_epoch = history_df.loc[history_df['fid_score'].idxmin()]
    print(f"\n[+] Best FID achieved: {best_epoch['fid_score']:.4f} at Epoch {int(best_epoch['epoch'])}")
    print("[+] Saved FID curve to: outputs/metrics/fid_curve.png")
else:
    print("[-] FID metric not found in training history.")

### Step 7: Inspect Milestone Sample Grids

In [ ]:
from PIL import Image
import glob

sample_files = sorted(glob.glob("outputs/samples/epoch_*_grid.png"))
for f in sample_files:
    print(f"Displaying: {f}")
    display(Image.open(f))

### Step 8: Bundle & Download Outputs for Local Streamlit Dashboard

In [ ]:
from google.colab import files
!zip -r ganforge_colab_outputs.zip outputs/ data/processed/
files.download('ganforge_colab_outputs.zip')
print("[+] Artifacts packaged and download triggered!")